In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 106.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct-1M")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct-1M",
                                             torch_dtype=torch.float16,
    																				 device_map="auto")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

### Загрузка текста

In [ ]:
file = 'ENG_article.txt'
with open(file, 'r', encoding='cp1251') as file:
    data = file.read().replace('\n', ' ')

In [ ]:
print(len(data))

8291


### Системный промпт и запрос пользователя

In [ ]:
messages = [
    {"role": "system", "content": "На основе данного текста дай ответы на все вопросы на русском языке. Если в тексте упоминается другой термин, обязательно используй именно его. Не заменяй слова из вопроса на синонимы, если в тексте они отличаются."},
    {"role": "user", "content": f"Текст: {data}"
    "Вопросы: В каком году была обозначена проблема взрывающихся градиентов? "
    "Кто в 1891 году разработал метод уничтожающей производной? "
    "Кто предложил цепное правило дифференцирования и в каком году?"}
]

Токенизируем промпты

In [ ]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	return_dict=True,
	return_tensors="pt",
).to("cuda")
print(inputs[0:160])

[Encoding(num_tokens=2213, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]


Генерируем ответ модели

In [ ]:
generated_ids = model.generate(
    **inputs,
    max_new_tokens=1024,
#    temperature=0.9,
#    top_k=50,
#    repetition_penalty=1.2
)

In [ ]:
generated_ids_ = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
]

Выводим ответ на экран

In [ ]:
response = tokenizer.batch_decode(generated_ids_, skip_special_tokens=True)[0]
print(response)

Вопросы:

1. В каком году была обозначена проблема взрывающихся градиентов?
Ответ: В 1991 году была обозначена проблема взрывающихся градиентов.

2. Кто в 1891 году разработал метод уничтожающей производной?
Ответ: Метод уничтожающей производной был разработан Готфридом Вильгельмом Либнitz в 1673 году.

3. Кто предложил цепное правило дифференцирования и в каком году?
Ответ: Цепное правило дифференцирования было предложено Готфридом Вильгельмом Либнitz в 1673 году.
